In [2]:
import polars as pl
from paths import ROOT_DIR, DATA_DIR

In [11]:
train_df = pl.read_ndjson(ROOT_DIR / "data/LibriSpeech-sint/manifest.jsonl")

In [12]:
train_df.columns

['audio_filepath',
 'text',
 'language',
 'condition',
 'duration',
 'sample_rate',
 'utterance_id']

In [13]:
train_df["duration"].describe()

statistic,value
str,f64
"""count""",114156.0
"""null_count""",0.0
"""mean""",12.688853
"""std""",3.574606
"""min""",1.41
"""25%""",11.62
"""50%""",13.995
"""75%""",15.165
"""max""",24.525


In [14]:
train_df["duration"].sum() / 60 / 60

402.3635166666665

In [15]:
import random


def create_stratified_dataset(
    df: pl.DataFrame, target_hours: float = 100.0
) -> pl.DataFrame:
    target_seconds = target_hours * 3600

    rows = df.iter_rows(named=True)
    rows = list(rows)
    random.shuffle(rows)

    selected = []
    total_seconds = 0.0
    for row in rows:
        if total_seconds >= target_seconds:
            break
        selected.append(row)
        total_seconds += row["duration"]

    result_df = pl.DataFrame(selected).select(
        [
            "audio_filepath",
            "text",
            "language",
            "condition",
            "duration",
            "sample_rate",
            "utterance_id",
        ]
    )

    print(f"Total duration: {total_seconds / 3600:.2f}h")
    print(f"Total rows: {len(result_df)}")
    print(
        f"Condition distribution:\n{result_df['condition'].value_counts().sort('condition')}"
    )

    return result_df


result = create_stratified_dataset(train_df, target_hours=100.0)

Total duration: 100.00h
Total rows: 28345
Condition distribution:
shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 7160  │
│ good      ┆ 7056  │
│ okay      ┆ 7061  │
│ perfect   ┆ 7068  │
└───────────┴───────┘


In [16]:
from pathlib import Path


def copy_and_write_manifest(
    df: pl.DataFrame,
    input_dir: str,
    output_dir: str,
    manifest_path: str,
) -> pl.DataFrame:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    audio_dir = output_dir / "audio"

    new_rows = []

    for row in tqdm(df.iter_rows(named=True), total=len(df), desc="Copying audio"):
        src = input_dir / row["audio_filepath"]

        condition_dir = audio_dir / row["condition"]
        condition_dir.mkdir(parents=True, exist_ok=True)

        dst = condition_dir / f"{row['utterance_id']}{src.suffix}"
        shutil.copy2(src, dst)

        row["audio_filepath"] = str(
            dst.relative_to(output_dir)
        )  # relative to output_dir
        new_rows.append(row)

    new_df = pl.DataFrame(new_rows).select(
        [
            "audio_filepath",
            "text",
            "language",
            "condition",
            "duration",
            "sample_rate",
            "utterance_id",
        ]
    )

    new_df.write_ndjson(manifest_path)
    print(f"Manifest written to: {manifest_path}")
    print(f"Audio files copied to: {audio_dir}")
    print(f"Total files: {len(new_df)}")
    print(f"Total duration: {new_df['duration'].sum() / 3600:.2f}h")

    return new_df


new_df = copy_and_write_manifest(
    df=result,
    input_dir=DATA_DIR / "LibriSpeech-sint",
    output_dir=DATA_DIR / "LibriSpeech-train-sim-100h",
    manifest_path=DATA_DIR / "LibriSpeech-train-sim-100h/manifest.jsonl",
)

Copying audio: 100%|██████████| 28345/28345 [00:19<00:00, 1457.82it/s]

Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-train-sim-100h/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-train-sim-100h/audio
Total files: 28345
Total duration: 100.00h


In [17]:
small_subset = create_stratified_dataset(train_df, target_hours=40)

new_df = copy_and_write_manifest(
    df=small_subset,
    input_dir=DATA_DIR / "LibriSpeech-sint",
    output_dir=DATA_DIR / "LibriSpeech-train-sim-40h",
    manifest_path=DATA_DIR / "LibriSpeech-train-sim-40h/manifest.jsonl",
)

Total duration: 40.00h
Total rows: 11317
Condition distribution:
shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 2780  │
│ good      ┆ 2805  │
│ okay      ┆ 2879  │
│ perfect   ┆ 2853  │
└───────────┴───────┘


Copying audio: 100%|██████████| 11317/11317 [00:05<00:00, 2101.78it/s]

Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-train-sim-40h/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-train-sim-40h/audio
Total files: 11317
Total duration: 40.00h


In [18]:
random_path = Path("/home/ezka/vhf-bak/data/ESC50-sint/manifest.jsonl")

In [19]:
random_df = pl.read_ndjson(random_path)

In [20]:
random_df = create_stratified_dataset(random_df, target_hours=0.2)

Total duration: 0.20h
Total rows: 144
Condition distribution:
shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 31    │
│ good      ┆ 42    │
│ okay      ┆ 30    │
│ perfect   ┆ 41    │
└───────────┴───────┘


In [23]:
copy_and_write_manifest(
    df=random_df,
    input_dir="/home/ezka/vhf-bak/data/ESC50-sint",
    output_dir="/home/ezka/vhf-bak/data/ESC50-sint-20m",
    manifest_path="/home/ezka/vhf-bak/data/ESC50-sint-20m/manifest.jsonl",
)

Copying audio: 100%|██████████| 144/144 [00:00<00:00, 1252.31it/s]

Manifest written to: /home/ezka/vhf-bak/data/ESC50-sint-20m/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/ESC50-sint-20m/audio
Total files: 144
Total duration: 0.20h


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/bad/2-103426-A-3.wav""","""""","""en""","""bad""",5.0,16000,"""2-103426-A-3"""
"""audio/okay/2-68595-A-15.wav""","""""","""en""","""okay""",5.0,16000,"""2-68595-A-15"""
"""audio/perfect/1-208757-B-2.wav""","""""","""en""","""perfect""",5.0,16000,"""1-208757-B-2"""
"""audio/perfect/2-60794-A-26.wav""","""""","""en""","""perfect""",5.0,16000,"""2-60794-A-26"""
"""audio/bad/1-40621-A-28.wav""","""""","""en""","""bad""",5.0,16000,"""1-40621-A-28"""
…,…,…,…,…,…,…
"""audio/bad/2-122763-A-29.wav""","""""","""en""","""bad""",5.0,16000,"""2-122763-A-29"""
"""audio/good/4-181708-A-32.wav""","""""","""en""","""good""",5.0,16000,"""4-181708-A-32"""
"""audio/bad/5-127990-A-2.wav""","""""","""en""","""bad""",5.0,16000,"""5-127990-A-2"""


In [1]:
from pathlib import Path
import polars as pl


def merge_datasets(
    dataset_dirs: list[str | Path],
    output_dir: str | Path,
) -> pl.DataFrame:
    output_dir = Path(output_dir)
    audio_dir = output_dir / "audio"
    manifest_path = output_dir / "manifest.jsonl"

    all_rows = []

    for dataset_dir in dataset_dirs:
        dataset_dir = Path(dataset_dir)
        manifest = dataset_dir / "manifest.jsonl"

        df = pl.read_ndjson(manifest)

        for row in tqdm(
            df.iter_rows(named=True), total=len(df), desc=f"Copying {dataset_dir.name}"
        ):
            src = dataset_dir / row["audio_filepath"]

            condition_dir = audio_dir / row["condition"]
            condition_dir.mkdir(parents=True, exist_ok=True)

            dst = condition_dir / f"{row['utterance_id']}{src.suffix}"

            # Handle collisions: if same utterance_id exists from another dataset
            if dst.exists():
                dst = (
                    condition_dir
                    / f"{dataset_dir.name}__{row['utterance_id']}{src.suffix}"
                )

            shutil.copy2(src, dst)

            row["audio_filepath"] = str(dst.relative_to(output_dir))
            all_rows.append(row)

    new_df = pl.DataFrame(all_rows).select(
        [
            "audio_filepath",
            "text",
            "language",
            "condition",
            "duration",
            "sample_rate",
            "utterance_id",
        ]
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    new_df.write_ndjson(manifest_path)

    print(f"Manifest written to: {manifest_path}")
    print(f"Audio files copied to: {audio_dir}")
    print(f"Total files: {len(new_df)}")
    print(f"Total duration: {new_df['duration'].sum() / 3600:.2f}h")
    print(f"Conditions: {new_df['condition'].value_counts().sort('condition')}")

    return new_df

In [25]:
merge_datasets(
    dataset_dirs=[DATA_DIR / "LibriSpeech-train-sim-100h", DATA_DIR / "ESC50-sint-20m"],
    output_dir=DATA_DIR / "LibriSpeech-train-final",
)

Copying ESC50-sint-20m: 100%|██████████| 144/144 [00:00<00:00, 7831.28it/s]


Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-train-final/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-train-final/audio
Total files: 28489
Total duration: 100.20h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 7191  │
│ good      ┆ 7098  │
│ okay      ┆ 7091  │
│ perfect   ┆ 7109  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/perfect/8425-291444-0013…","""THE INFANT YEARS OF OUR CITY T…","""en""","""perfect""",15.895,16000,"""8425-291444-0013"""
"""audio/perfect/3807-4955-0000.w…","""CHAPTER TWENTY FIVE THE FLIGHT…","""en""","""perfect""",15.105,16000,"""3807-4955-0000"""
"""audio/okay/481-123719-0006.wav""","""THERE WERE MOMENTS OF SUCH POS…","""en""","""okay""",15.61,16000,"""481-123719-0006"""
"""audio/okay/89-218-0054.wav""","""TO CLOSE HER EYES IN SLEEP THA…","""en""","""okay""",15.29,16000,"""89-218-0054"""
"""audio/perfect/3526-176653-0008…","""BUT STOOD STILL WINDING SOMETH…","""en""","""perfect""",15.695,16000,"""3526-176653-0008"""
…,…,…,…,…,…,…
"""audio/bad/2-122763-A-29.wav""","""""","""en""","""bad""",5.0,16000,"""2-122763-A-29"""
"""audio/good/4-181708-A-32.wav""","""""","""en""","""good""",5.0,16000,"""4-181708-A-32"""
"""audio/bad/5-127990-A-2.wav""","""""","""en""","""bad""",5.0,16000,"""5-127990-A-2"""


In [26]:
merge_datasets(
    dataset_dirs=[DATA_DIR / "LibriSpeech-train-sim-40h", DATA_DIR / "ESC50-sint-20m"],
    output_dir=DATA_DIR / "LibriSpeech-train-small",
)

Copying ESC50-sint-20m: 100%|██████████| 144/144 [00:00<00:00, 7501.08it/s]

Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-train-small/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-train-small/audio
Total files: 11461
Total duration: 40.20h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 2811  │
│ good      ┆ 2847  │
│ okay      ┆ 2909  │
│ perfect   ┆ 2894  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/bad/1963-142776-0011.wav""","""SEASONABLE FROM JUNE TO MARCH …","""en""","""bad""",14.86,16000,"""1963-142776-0011"""
"""audio/okay/311-124404-0083.wav""","""WHENCE IT HAPPENS THAT IF THE …","""en""","""okay""",14.175,16000,"""311-124404-0083"""
"""audio/perfect/4397-15668-0034.…","""FOR TO IT HUNDREDS OF THEM OWE…","""en""","""perfect""",13.7,16000,"""4397-15668-0034"""
"""audio/perfect/5322-7680-0023.w…","""TAKE IT TAKE IT IF YOU DON'T Y…","""en""","""perfect""",13.61,16000,"""5322-7680-0023"""
"""audio/okay/4898-28461-0017.wav""","""THE RIVER REJOICING IN ITS STR…","""en""","""okay""",16.14,16000,"""4898-28461-0017"""
…,…,…,…,…,…,…
"""audio/bad/2-122763-A-29.wav""","""""","""en""","""bad""",5.0,16000,"""2-122763-A-29"""
"""audio/good/4-181708-A-32.wav""","""""","""en""","""good""",5.0,16000,"""4-181708-A-32"""
"""audio/bad/5-127990-A-2.wav""","""""","""en""","""bad""",5.0,16000,"""5-127990-A-2"""


In [9]:
merge_datasets(
    dataset_dirs=[
        DATA_DIR / "LibriSpeech-dev-clean-sint",
        DATA_DIR / "LibriSpeech-dev-other-sint",
    ],
    output_dir=DATA_DIR / "LibriSpeech-dev-combined",
)

Copying LibriSpeech-dev-other-sint: 100%|██████████| 11456/11456 [00:01<00:00, 7326.53it/s]

Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-dev-combined/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-dev-combined/audio
Total files: 22268
Total duration: 42.04h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 5567  │
│ good      ┆ 5567  │
│ okay      ┆ 5567  │
│ perfect   ┆ 5567  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/good/1272-128104-0014.wa…","""BY HARRY QUILTER M A""","""en""","""good""",2.245,16000,"""1272-128104-0014"""
"""audio/okay/1272-128104-0014.wa…","""BY HARRY QUILTER M A""","""en""","""okay""",2.245,16000,"""1272-128104-0014"""
"""audio/bad/1272-128104-0014.wav""","""BY HARRY QUILTER M A""","""en""","""bad""",2.245,16000,"""1272-128104-0014"""
"""audio/perfect/1272-128104-0014…","""BY HARRY QUILTER M A""","""en""","""perfect""",2.245,16000,"""1272-128104-0014"""
"""audio/good/1272-128104-0008.wa…","""AS FOR ETCHINGS THEY ARE OF TW…","""en""","""good""",5.12,16000,"""1272-128104-0008"""
…,…,…,…,…,…,…
"""audio/perfect/8288-274162-0035…","""I NEVER COMPLAIN AS YOU KNOW B…","""en""","""perfect""",15.73,16000,"""8288-274162-0035"""
"""audio/good/8288-274150-0000.wa…","""AND AGAIN WHEN THE ARTIST FOLL…","""en""","""good""",26.105,16000,"""8288-274150-0000"""
"""audio/okay/8288-274150-0000.wa…","""AND AGAIN WHEN THE ARTIST FOLL…","""en""","""okay""",26.105,16000,"""8288-274150-0000"""


In [10]:
merge_datasets(
    dataset_dirs=[
        DATA_DIR / "LibriSpeech-test-clean-sint",
        DATA_DIR / "LibriSpeech-test-other-sint",
    ],
    output_dir=DATA_DIR / "LibriSpeech-test-combined",
)

Copying LibriSpeech-test-other-sint: 100%|██████████| 11756/11756 [00:02<00:00, 3974.38it/s]

Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-test-combined/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-test-combined/audio
Total files: 22236
Total duration: 42.98h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 5559  │
│ good      ┆ 5559  │
│ okay      ┆ 5559  │
│ perfect   ┆ 5559  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/good/1089-134686-0014.wa…","""HE TRIED TO THINK HOW IT COULD…","""en""","""good""",2.225,16000,"""1089-134686-0014"""
"""audio/okay/1089-134686-0014.wa…","""HE TRIED TO THINK HOW IT COULD…","""en""","""okay""",2.225,16000,"""1089-134686-0014"""
"""audio/bad/1089-134686-0014.wav""","""HE TRIED TO THINK HOW IT COULD…","""en""","""bad""",2.225,16000,"""1089-134686-0014"""
"""audio/perfect/1089-134686-0014…","""HE TRIED TO THINK HOW IT COULD…","""en""","""perfect""",2.225,16000,"""1089-134686-0014"""
"""audio/good/1089-134686-0003.wa…","""HELLO BERTIE ANY GOOD IN YOUR …","""en""","""good""",2.68,16000,"""1089-134686-0003"""
…,…,…,…,…,…,…
"""audio/perfect/8461-281231-0030…","""THE TEMPLAR IS FLED SAID DE BR…","""en""","""perfect""",16.69,16000,"""8461-281231-0030"""
"""audio/good/8461-281231-0024.wa…","""HE LEFT THE GALLANT BAND OF FO…","""en""","""good""",20.21,16000,"""8461-281231-0024"""
"""audio/okay/8461-281231-0024.wa…","""HE LEFT THE GALLANT BAND OF FO…","""en""","""okay""",20.21,16000,"""8461-281231-0024"""


In [24]:
import json

atc_train_path = Path(
    "/Users/ezka/PycharmProjects/VHF-ASR/data/ESC50_vhf_simulated-20m"
)

atc_train_manifest_data = []
with open(atc_train_path / "manifest.jsonl", "r") as f:
    for line in f:
        atc_train_manifest_data.append(json.loads(line))

In [25]:
with open(
    "/Users/ezka/PycharmProjects/VHF-ASR/data/LibriSpeech-train-sim-100h/manifest.jsonl",
    "a",
) as f:
    for entry in atc_train_manifest_data:
        f.write(json.dumps(entry) + "\n")

In [26]:
import polars as pl
from paths import ROOT_DIR

train_df = pl.read_ndjson(ROOT_DIR / "data/LibriSpeech-train-sim-100h/manifest.jsonl")

In [27]:
train_df["duration"].sum() / 60 / 60

106.09631999999996

In [4]:
from pathlib import Path
from tqdm import tqdm
import polars as pl
import shutil
from paths import DATA_DIR


def merge_datasets(
    dataset_dirs: list[str | Path],
    output_dir: str | Path,
    manifests: list[str] | str = "manifest.jsonl",
) -> pl.DataFrame:
    output_dir = Path(output_dir)
    audio_dir = output_dir / "audio"
    manifest_path = output_dir / "manifest.jsonl"

    if isinstance(manifests, str):
        manifests = [manifests] * len(dataset_dirs)

    if len(manifests) != len(dataset_dirs):
        raise ValueError(
            f"manifests has {len(manifests)} entries but dataset_dirs has "
            f"{len(dataset_dirs)} — lengths must match."
        )

    all_rows = []

    for dataset_dir, manifest_name in zip(dataset_dirs, manifests):
        dataset_dir = Path(dataset_dir)
        manifest = dataset_dir / manifest_name

        if not manifest.exists():
            raise FileNotFoundError(f"Manifest not found: {manifest}")

        df = pl.read_ndjson(manifest)

        for row in tqdm(
            df.iter_rows(named=True),
            total=len(df),
            desc=f"Copying {dataset_dir.name} ({manifest_name})",
        ):
            src = dataset_dir / row["audio_filepath"]

            condition_dir = audio_dir / row["condition"]
            condition_dir.mkdir(parents=True, exist_ok=True)

            dst = condition_dir / f"{row['utterance_id']}{src.suffix}"

            if dst.exists():
                dst = (
                    condition_dir
                    / f"{dataset_dir.name}__{row['utterance_id']}{src.suffix}"
                )

            shutil.copy2(src, dst)

            row["audio_filepath"] = str(dst.relative_to(output_dir))
            all_rows.append(row)

    new_df = pl.DataFrame(all_rows).select(
        [
            "audio_filepath",
            "text",
            "language",
            "condition",
            "duration",
            "sample_rate",
            "utterance_id",
        ]
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    new_df.write_ndjson(manifest_path)

    print(f"Manifest written to: {manifest_path}")
    print(f"Audio files copied to: {audio_dir}")
    print(f"Total files: {len(new_df)}")
    print(f"Total duration: {new_df['duration'].sum() / 3600:.2f}h")
    print(f"Conditions: {new_df['condition'].value_counts().sort('condition')}")

    return new_df

In [5]:
merge_datasets(
    dataset_dirs=[DATA_DIR / "LibriSpeech-train-final", DATA_DIR / "atc_asr_train"],
    manifests=["manifest.jsonl", "manifest_ecapa.jsonl"],
    output_dir=DATA_DIR / "LibriSpeech-train-final-with-atc",
)

Copying atc_asr_train (manifest_ecapa.jsonl): 100%|██████████| 6497/6497 [00:00<00:00, 9372.88it/s]


Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-train-final-with-atc/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-train-final-with-atc/audio
Total files: 34986
Total duration: 106.10h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 7218  │
│ good      ┆ 7269  │
│ okay      ┆ 7220  │
│ perfect   ┆ 13279 │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/perfect/8425-291444-0013…","""THE INFANT YEARS OF OUR CITY T…","""en""","""perfect""",15.895,16000,"""8425-291444-0013"""
"""audio/perfect/3807-4955-0000.w…","""CHAPTER TWENTY FIVE THE FLIGHT…","""en""","""perfect""",15.105,16000,"""3807-4955-0000"""
"""audio/okay/481-123719-0006.wav""","""THERE WERE MOMENTS OF SUCH POS…","""en""","""okay""",15.61,16000,"""481-123719-0006"""
"""audio/okay/89-218-0054.wav""","""TO CLOSE HER EYES IN SLEEP THA…","""en""","""okay""",15.29,16000,"""89-218-0054"""
"""audio/perfect/3526-176653-0008…","""BUT STOOD STILL WINDING SOMETH…","""en""","""perfect""",15.695,16000,"""3526-176653-0008"""
…,…,…,…,…,…,…
"""audio/perfect/ZZO15U50FH61N7FD…",null,"""en""","""perfect""",2.94,16000,"""ZZO15U50FH61N7FDRFQX"""
"""audio/perfect/ZZRJO7VBHH5OL5U7…",null,"""en""","""perfect""",2.35,16000,"""ZZRJO7VBHH5OL5U7NTED"""
"""audio/perfect/ZZSFZ2ZR9GHXZQUH…",null,"""en""","""perfect""",3.06,16000,"""ZZSFZ2ZR9GHXZQUHN8L6"""


In [6]:
merge_datasets(
    dataset_dirs=[DATA_DIR / "LibriSpeech-dev-combined", DATA_DIR / "atc_asr_val"],
    manifests=["manifest.jsonl", "manifest_ecapa.jsonl"],
    output_dir=DATA_DIR / "LibriSpeech-dev-combined-with-atc",
)

Copying LibriSpeech-dev-combined (manifest.jsonl): 100%|██████████| 22268/22268 [00:03<00:00, 7276.51it/s]
Copying atc_asr_val (manifest_ecapa.jsonl): 100%|██████████| 812/812 [00:00<00:00, 9517.59it/s]


Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-dev-combined-with-atc/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-dev-combined-with-atc/audio
Total files: 23080
Total duration: 42.78h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 5574  │
│ good      ┆ 5584  │
│ okay      ┆ 5586  │
│ perfect   ┆ 6336  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/good/1272-128104-0014.wa…","""BY HARRY QUILTER M A""","""en""","""good""",2.245,16000,"""1272-128104-0014"""
"""audio/okay/1272-128104-0014.wa…","""BY HARRY QUILTER M A""","""en""","""okay""",2.245,16000,"""1272-128104-0014"""
"""audio/bad/1272-128104-0014.wav""","""BY HARRY QUILTER M A""","""en""","""bad""",2.245,16000,"""1272-128104-0014"""
"""audio/perfect/1272-128104-0014…","""BY HARRY QUILTER M A""","""en""","""perfect""",2.245,16000,"""1272-128104-0014"""
"""audio/good/1272-128104-0008.wa…","""AS FOR ETCHINGS THEY ARE OF TW…","""en""","""good""",5.12,16000,"""1272-128104-0008"""
…,…,…,…,…,…,…
"""audio/perfect/ZKWSSNML4C7F4YGF…",null,"""en""","""perfect""",0.75,16000,"""ZKWSSNML4C7F4YGF76ST"""
"""audio/perfect/ZNKQ5A36E1JQQXF2…",null,"""en""","""perfect""",2.49,16000,"""ZNKQ5A36E1JQQXF2A8JC"""
"""audio/perfect/ZQ1VKVV411QIZRMT…",null,"""en""","""perfect""",2.71,16000,"""ZQ1VKVV411QIZRMT0NHB"""


In [7]:
merge_datasets(
    dataset_dirs=[DATA_DIR / "LibriSpeech-test-combined", DATA_DIR / "atc_asr_test"],
    manifests=["manifest.jsonl", "manifest_ecapa.jsonl"],
    output_dir=DATA_DIR / "LibriSpeech-test-combined-with-atc",
)

Copying LibriSpeech-test-combined (manifest.jsonl): 100%|██████████| 22236/22236 [00:03<00:00, 6783.21it/s]
Copying atc_asr_test (manifest_ecapa.jsonl): 100%|██████████| 813/813 [00:00<00:00, 8917.14it/s]


Manifest written to: /home/ezka/vhf-bak/data/LibriSpeech-test-combined-with-atc/manifest.jsonl
Audio files copied to: /home/ezka/vhf-bak/data/LibriSpeech-test-combined-with-atc/audio
Total files: 23049
Total duration: 43.74h
Conditions: shape: (4, 2)
┌───────────┬───────┐
│ condition ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ bad       ┆ 5560  │
│ good      ┆ 5576  │
│ okay      ┆ 5574  │
│ perfect   ┆ 6339  │
└───────────┴───────┘


audio_filepath,text,language,condition,duration,sample_rate,utterance_id
str,str,str,str,f64,i64,str
"""audio/good/1089-134686-0014.wa…","""HE TRIED TO THINK HOW IT COULD…","""en""","""good""",2.225,16000,"""1089-134686-0014"""
"""audio/okay/1089-134686-0014.wa…","""HE TRIED TO THINK HOW IT COULD…","""en""","""okay""",2.225,16000,"""1089-134686-0014"""
"""audio/bad/1089-134686-0014.wav""","""HE TRIED TO THINK HOW IT COULD…","""en""","""bad""",2.225,16000,"""1089-134686-0014"""
"""audio/perfect/1089-134686-0014…","""HE TRIED TO THINK HOW IT COULD…","""en""","""perfect""",2.225,16000,"""1089-134686-0014"""
"""audio/good/1089-134686-0003.wa…","""HELLO BERTIE ANY GOOD IN YOUR …","""en""","""good""",2.68,16000,"""1089-134686-0003"""
…,…,…,…,…,…,…
"""audio/perfect/ZRVW14Z2FO4EZP98…",null,"""en""","""perfect""",3.56,16000,"""ZRVW14Z2FO4EZP98Y33S"""
"""audio/perfect/ZUKPIHDOAU3FN9X1…",null,"""en""","""perfect""",3.0,16000,"""ZUKPIHDOAU3FN9X1BS2Z"""
"""audio/perfect/ZXGUAKVXT2FHO8CC…",null,"""en""","""perfect""",2.36,16000,"""ZXGUAKVXT2FHO8CC98E3"""
